<a href="https://colab.research.google.com/github/maannasviiii/BIS-LAB/blob/main/Week3_ACO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random

class AntColonyTSP:
    def __init__(self, cities, n_ants, n_iterations, alpha=1.0, beta=2.0, rho=0.5, q=100, initial_pheromone=0.1):
        # 1. Define the Problem & 2. Initialize Parameters
        self.cities = np.array(cities)
        self.n_cities = len(cities)
        self.n_ants = n_ants
        self.n_iterations = n_iterations
        self.alpha = alpha          # Importance of pheromone
        self.beta = beta            # Importance of heuristic (1 / distance)
        self.rho = rho              # Evaporation rate
        self.q = q                  # Pheromone deposit factor

        # Precompute distance and heuristic matrices
        self.distances = self._compute_distances()
        self.pheromones = np.full((self.n_cities, self.n_cities), initial_pheromone)

        # Avoid division by zero on diagonal
        with np.errstate(divide='ignore'):
            self.heuristic = 1.0 / self.distances
            self.heuristic[self.distances == 0] = 0

    def _compute_distances(self):
        # Euclidean distance between every pair of cities
        diff = self.cities[:, np.newaxis, :] - self.cities[np.newaxis, :, :]
        return np.sqrt(np.sum(diff ** 2, axis=-1))

    def _select_next_city(self, current_city, unvisited):
        # Calculate numerator: (tau^alpha) * (eta^beta)
        pheromone = self.pheromones[current_city, unvisited] ** self.alpha
        heuristic = self.heuristic[current_city, unvisited] ** self.beta
        probabilities = pheromone * heuristic

        # Normalize to create a probability distribution
        prob_sum = np.sum(probabilities)
        if prob_sum == 0:
            return random.choice(unvisited)

        probabilities /= prob_sum
        return np.random.choice(unvisited, p=probabilities)

    def _construct_tour(self):
        # 3. Construct Solutions for a single ant
        start_city = random.randint(0, self.n_cities - 1)
        tour = [start_city]
        unvisited = list(set(range(self.n_cities)) - {start_city})

        current_city = start_city
        while unvisited:
            next_city = self._select_next_city(current_city, unvisited)
            tour.append(next_city)
            unvisited.remove(next_city)
            current_city = next_city

        return tour

    def _calculate_tour_distance(self, tour):
        distance = 0.0
        for i in range(self.n_cities):
            from_city = tour[i]
            to_city = tour[(i + 1) % self.n_cities]  # Wrap around to origin city
            distance += self.distances[from_city, to_city]
        return distance

    def _update_pheromones(self, all_tours):
        # 4. Update Pheromones (Evaporation + Deposition)
        self.pheromones *= (1.0 - self.rho)

        for tour, distance in all_tours:
            deposit = self.q / distance
            for i in range(self.n_cities):
                u = tour[i]
                v = tour[(i + 1) % self.n_cities]
                self.pheromones[u, v] += deposit
                self.pheromones[v, u] += deposit

    def solve(self):
        best_tour = None
        best_distance = float('inf')

        # 5. Iterate
        for iteration in range(self.n_iterations):
            all_tours = []
            for _ in range(self.n_ants):
                tour = self._construct_tour()
                distance = self._calculate_tour_distance(tour)
                all_tours.append((tour, distance))

                if distance < best_distance:
                    best_distance = distance
                    best_tour = tour

            self._update_pheromones(all_tours)

        # 6. Output the Best Solution
        return best_tour, best_distance


# --- Example Execution ---
if __name__ == "__main__":
    # Step 1: Define 10 cities with (x, y) coordinates
    city_coordinates = [
        (0, 0), (1, 5), (4, 2), (6, 6), (8, 3),
        (3, 7), (7, 1), (2, 4), (9, 8), (5, 5)
    ]

    # Step 2: Initialize parameters and run solver
    aco = AntColonyTSP(
        cities=city_coordinates,
        n_ants=20,
        n_iterations=100,
        alpha=1.0,
        beta=2.5,
        rho=0.5,
        q=100,
        initial_pheromone=0.1
    )

    best_route, min_distance = aco.solve()

    # Format route to show return to origin
    full_route = best_route + [best_route[0]]
    print(f"Best Route (City Indices): {' -> '.join(map(str, full_route))}")
    print(f"Shortest Total Distance: {min_distance:.2f}")


Best Route (City Indices): 0 -> 7 -> 1 -> 5 -> 9 -> 3 -> 8 -> 4 -> 6 -> 2 -> 0
Shortest Total Distance: 31.53
